# Notebook rendering check

Inline `.ipynb` rendering landed on 2026-08-21. Open this notebook in Suzuri and
walk the checklist. Linked from [[grinding-the-ink]]; the script twin is
`python-rendering-check.py`, beside it in `code/`.

- [ ] Section 1 shows the `In [n]:` execution count beside every run cell
- [ ] Section 2 typesets LaTeX in markdown cells, not just in `.md` notes
- [ ] Section 3 renders a PNG output inline at its natural size
- [ ] Section 4 renders an HTML table output as a table, not as raw tags
- [ ] Section 5 shows a traceback in red, with the ANSI codes stripped
- [ ] Section 6 scrolls long stream output instead of stretching the page
- [ ] Section 7 leaves an unexecuted cell blank, with no output block
- [ ] Section 8 resolves a vault-relative image from a markdown cell

## 1. Execution counts and stdout


In [1]:
import math
from dataclasses import dataclass

SAMPLES = [(1.0, 0.21), (2.0, 0.38), (4.0, 0.62), (6.0, 0.78),
           (8.0, 0.88), (12.0, 0.99), (16.0, 1.04), (24.0, 1.08)]

print(f"{len(SAMPLES)} samples, {SAMPLES[0][0]:.0f}-{SAMPLES[-1][0]:.0f} min")
print("stone:", "Duan硯", "· 产地广东 · 等级1甲")


8 samples, 1-24 min
stone: Duan硯 · 产地广东 · 等级1甲


## 2. Math in a markdown cell

Density saturates, so we fit $D(t) = D_\infty\,(1 - e^{-t/\tau})$ with
$D_\infty$ in absorbance units and $\tau$ in minutes. The grid search minimises

$$\mathrm{SSE}(\tau) = \sum_{i=1}^{n} \bigl(D_i - \hat{D}_\infty(\tau)\,[1 - e^{-t_i/\tau}]\bigr)^2,
\qquad
\hat{D}_\infty(\tau) = \frac{\sum_i D_i b_i}{\sum_i b_i^2},\quad b_i = 1 - e^{-t_i/\tau}.$$

Half-saturation lands at $t_{50} = \tau \ln 2$. Ratios live in [[duan-ratios]];
the source is [@tanaka2019, p. 41].

## 3. A closed-form solve per candidate


In [2]:
@dataclass(frozen=True)
class Fit:
    d_inf: float
    tau: float
    sse: float

    @property
    def half_time(self) -> float:
        return self.tau * math.log(2)


def fit(data=SAMPLES):
    best = None
    for tau in (t / 10 for t in range(5, 400)):
        b = [1.0 - math.exp(-t / tau) for t, _ in data]
        d_inf = sum(bi * d for bi, (_, d) in zip(b, data)) / sum(bi * bi for bi in b)
        sse = sum((d - d_inf * bi) ** 2 for bi, (_, d) in zip(b, data))
        if best is None or sse < best.sse:
            best = Fit(d_inf, tau, sse)
    return best


f = fit()
f


Fit(d_inf=1.0796225708484396, tau=4.7, sse=0.00014903245930697457)


## 4. An image output


In [3]:
from IPython.display import Image, display

# The fitted curve over the eight samples, drawn on the brand's paper cream.
display(Image(filename="curve.png"))


## 5. An HTML table output


In [4]:
summary_table()  # one row per stone on the bench


## 6. A traceback


In [5]:
fit([(0.0, 0.0)])  # a single sample at t=0 makes the basis all-zero


ZeroDivisionError: float division by zero

## 7. Long and interleaved stream output


In [6]:
import sys

print("warming the stone...", file=sys.stderr)
for t, d in SAMPLES:
    hat = f.d_inf * (1 - math.exp(-t / f.tau))
    print(f"t={t:5.1f} min  D={d:.2f}  fit={hat:.3f}  "
          f"resid={d - hat:+.4f}  {'█' * round(hat * 40)}")
print(f"t50={f.half_time:.2f} min, sse={f.sse:.2e}")


warming the stone...


t=  1.0 min  D=0.21  fit=0.209  resid=+0.0006  ████████
t=  2.0 min  D=0.38  fit=0.379  resid=+0.0007  ███████████████
t=  4.0 min  D=0.62  fit=0.616  resid=+0.0035  █████████████████████████
t=  6.0 min  D=0.78  fit=0.769  resid=+0.0107  ███████████████████████████████
t=  8.0 min  D=0.88  fit=0.867  resid=+0.0128  ███████████████████████████████████
t= 12.0 min  D=0.99  fit=0.996  resid=-0.0056  ████████████████████████████████████████
t= 16.0 min  D=1.04  fit=1.045  resid=-0.0053  ██████████████████████████████████████████
t= 24.0 min  D=1.08  fit=1.075  resid=+0.0053  ███████████████████████████████████████████
t50=3.26 min, sse=1.49e-04


## 8. An unexecuted cell

Nothing has been run below — no `In [n]:`, no output block.


In [ ]:
# Re-fit against the She stone once its plates are digitised.
fit(SHE_SAMPLES)


## 9. A vault-relative image from a notebook

The same attachment `grinding-the-ink.md` embeds, addressed from `code/`:

![suzuri brand sheet — flat mark, app icons, palette|452](../md/attachments/suzuri-banner.png)

Back to [[grinding-the-ink]] · tables in [[table-rendering-check]] ·
math in [[math-rendering-check]].
